# Tilt Correction Tutorial 1

This tutorial demonstrates 2D tilt correction (rectification) using three arc lamp
spectra (HgAr, Ne, and Xe) and a single long-slit science frame, all observed with the
[Osiris spectrograph](https://www.gtc.iac.es/instruments/osiris/) at the
[Gran Telescopio Canarias (GTC)](https://www.gtc.iac.es/) using the R1000R grism
configuration in 2012.

OSIRIS, which operated until 2023 before being upgraded to OSIRIS+, featured two 2048 x 4096
pixel Marconi CCD detectors. For simplicity and file size considerations, we use data from just
 one CCD that has been binned to 512 x 1024 pixels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from astropy.visualization import simple_norm
from specreduce.tilt_correction import TiltCorrection
from common import read_data

plt.rc('figure', figsize=(6.3, 2))
plt.rc('font', size=8)

## 1. Read in the Arc Frames and Object Frame

We have hidden the data reading in `common.read_data` utility function as not to distract from
the main topic. The function returns a tuple of three arc frames as bias-subtracted `CCDData`
instances, a tuple containing the arc lamp names, and a single bias-subtracted `CCDData` instance
containing the science data. (We are cutting some corners what comes to basic data reduction and
skipping the steps not absolutely necessary for tilt correction, such as flat field correction,
but normally you would want to do these basic steps before.)

Let's read in the data and take a look at it.

In [ ]:
arcs, lamps, obj = read_data()
frames = arcs + [obj]
labels = lamps + ('Target',)

In [ ]:
fig, axs = plt.subplots(4, figsize=(6.3, 5), sharex='all', constrained_layout=True)
for i,d in enumerate(frames):
    axs[i].imshow(d.data, origin='lower', aspect='auto', cmap=plt.cm.Blues,
                  norm=simple_norm(d.data, stretch='log', vmin=0, vmax=250_000))
    axs[i].text(0.01, 0.9, labels[i], va='top', ha='left', c='k', transform=axs[i].transAxes)
plt.setp(axs, ylabel='CD axis [pix]')
plt.setp(axs[-1], xlabel='Dispersion axis [pix]');

We immediately see that OSIRIS spectra exhibit significant tilt distortion that must be corrected
to achieve reliable background subtraction. The example science frame comes from a time-series
of spectroscopic observations taken during an exoplanet transit (transmission spectroscopy). The
scientific importance of proper tilt correction is highlighted by the original analysis of this
dataset [(Parviainen et al. 2016)](https://ui.adsabs.harvard.edu/abs/2016A%26A...585A.114P/abstract):
insufficient correction of the tilt distortion led to residual time variations in
telluric absorption lines that compromised the scientific results.


## 2. Initialize the Tilt Correction Class

The `TiltCorrection` class is initialized with the following parameters:

- ``ref_pixel``: Reference pixel coordinate near detector center. The polynomial
  fit centers around this pixel to improve numerical stability.
- ``arc_frames``: Either a single `CCDData` instance or a list of `CCDData` instances containing
  the arc lamp spectra. These should be bias-subtracted and include uncertainties for the
  internal line finding routine to work reliably.
- ``cd_sample_lims``: Pixel range for sampling the cross-dispersion direction.
- ``n_cd_samples``: Number of points to sample along cross-dispersion.
- ``cd_samples``: A list of cross-dispersion locations to use. Overrides ``n_cd_samples`` if provided.

The initialization sets up the transformation model that will be used to map tilted spectral features to straight lines.

In [ ]:
s = TiltCorrection(arcs, 300, n_cdisp_samples=7, cdisp_sample_lims=(50, 512))

## 3. Find Arc Lines

The `find_arc_lines()` method is used after initializing the `TiltCorrection` object to detect emission peaks in arc lamp spectra. The detection process employs the `specutils.fitting.find_lines_threshold` routine, which uses parameters for the expected line full-width half-maximum (FWHM) and noise threshold to identify significant emission lines above the background.

The process begins by identifying lines at the reference row, which are stored separately. The
method then detects lines at specified cross-dispersion locations to generate a set of 2D points
in the *detector space*. These detector space points are essential for mapping the spectral tilt and curvature patterns across the detector. The points are stored in a kd-tree for each arc frame and will be used in the model fitting process.

The *rectified space* is defined by the points found in the reference row. The 2D rectification process maps the *detector space* columns to align with the wavelengths in the reference row across all elements (rows) of the cross-dispersion axis.


In [ ]:
s.find_arc_lines(fwhm=2.5, noise_factor=10)

Let's take a look at the data to see how we track the spectral lines using both reference-row
samples and detector-space samples across our arc frames. By using kd-trees, we can match up
spectral lines successfully even if they're not found at every cross-dispersion sample point.
This approach helps us get a solid final fit, handling any outliers that pop up in the data.


In [ ]:
fig, axs = plt.subplots(3, figsize=(6.3, 4), sharex='all', constrained_layout=True)
for i,d in enumerate(arcs):
    axs[i].imshow(d.data, origin='lower', aspect='auto', cmap=plt.cm.Blues,
                  norm=simple_norm(d.data, stretch='log', vmin=0, vmax=250_000))
    axs[i].text(0.01, 0.9, lamps[i], va='top', ha='left', c='k', transform=axs[i].transAxes)
    axs[i].plot(s._samples_det_x[i], s._samples_det_y[i], 'k.', ms=3)
    axs[i].plot(s._lines_ref[i], np.full_like(s._lines_ref[i], s.ref_pixel[0]), 'r.', ms=3)
plt.setp(axs, ylabel='CD axis [pix]')
plt.setp(axs[-1], xlabel='Dispersion axis [pix]');

## 4. Fit the model

The `fit()` method is used to calculate the geometric transformation that maps tilted spectral features to straight lines. This method:

1. Takes a `degree` parameter (set to 4 here) that determines the polynomial order of the transformation
2. Uses the arc line positions detected earlier as reference points
3. Performs an initial optimization to find a rough transformation
4. Refines the solution through further optimization
5. Generates the final transformation model that will be used to rectify the spectra

The fitted model captures both the tilt and any curvature present in the spectral features across the detector.

In [ ]:
s.fit(degree=4)

Let's examine how well our transformation fits the data using the `TiltCorrection.plot_fit_quality` method. This visualization displays both 2D and 1D residuals between the measured detector-space points and the transformed reference points.

Our fit shows good accuracy across the detector. While the data contains some outlier points, they don't notably affect the overall fit quality in this example. If outliers do become problematic, we can enhance the transformation model by using `TiltCorrection.refine_fit()` with a tighter `match_distance_bound` parameter (such as 0.5 pixels). This refinement would focus the fit on only the most reliable emission line positions, ensuring better accuracy in the final transformation.


In [ ]:
s.plot_fit_quality(figsize=(6.3, 5), rlim=(-0.5, 0.5))

You can see the transform directly in detector space with the `TiltCorrection.plot_wavelength_contours` method. This method creates an informative visualization by overlaying contour lines showing where wavelengths remain constant across the detector surface.


In [ ]:
fig, axs = plt.subplots(4, figsize=(6.3, 5), sharex='all', constrained_layout=True)
for i,d in enumerate(frames):
    axs[i].imshow(d.data, origin='lower', aspect='auto', cmap=plt.cm.Blues,
                  norm=simple_norm(d.data, stretch='log', vmin=0, vmax=250_000))
    axs[i].text(0.01, 0.9, labels[i], va='top', ha='left', c='k', transform=axs[i].transAxes)
    s.plot_wavelength_contours(ax=axs[i])
plt.setp(axs, xlim=(0, arcs[0].shape[1]), ylabel='CD axis [pix]');
plt.setp(axs[-1], xlabel='Dispersion axis [pix]');

## 5. Rectify the frames

Now that we have our fitted transformation model, we can rectify the spectral frames with the `TiltCorrection.rectify()` method. The process is straightforward - the method takes a CCData object containing the input frame data and applies the fitted geometric transformation to rebin the pixels. This rebinning uses an exact flux-conserving approach to ensure data quality. The output is a new CCDData object where all spectral features line up with the detector rows and columns.

In [ ]:
rectified_data = [s.solution.resample(d.data) for d in frames]

In [ ]:
fig, axs = plt.subplots(4, figsize=(6.3, 5), sharex='all', constrained_layout=True)
for i,d in enumerate(rectified_data):
    axs[i].imshow(d.data, origin='lower', aspect='auto', cmap=plt.cm.Blues,
                  norm=simple_norm(d.data, stretch='log', vmin=0, vmax=150_000))
    axs[i].text(0.01, 0.9, labels[i], va='top', ha='left', c='k', transform=axs[i].transAxes)
    axs[i].grid(which='both', axis='x')
    axs[i].set_xticks(np.linspace(10, 1000, 19))
plt.setp(axs, ylabel='CD axis [pix]')
plt.setp(axs[-1], xlabel='Dispersion axis [pix]');

Let's check how well our rectification worked by comparing emission line positions. Here we overlay the reference row spectrum from the original arc frame with spectra sampled from different rows in the rectified data. You can see the spectral features line up beautifully across various rows after rectification - the emission lines match with sub-pixel precision, showing our tilt correction worked perfectly.


In [ ]:
fig, ax = plt.subplots(1, figsize=(6.3, 2), sharex='all', constrained_layout=True)

ref_row = s.ref_pixel[0]
rec_rows = [s.ref_pixel[0], 150, 450]

ax.plot(arcs[0].data[ref_row])
for i, r in enumerate(rec_rows):
    ax.plot(50_000*(i+1) + rectified_data[0].data[r])
plt.setp(ax, xlabel='Dispersion axis [Pix]', yticks=[])
plt.setp(ax, xlim=(50, 200));